# Two-Step Adams–Bashforth Method

Two-Step Adams–Bashforth Method 不额外查询中点，而是使用当前和上一步已缓存速度。它属于显式线性多步法：第一步没有历史速度，必须以 Explicit Euler Method warm-up；此后每一步只需 1 个新 NFE。

均匀步长 $h$ 下，对 $v(t)$ 做线性插值并积分：

$$
p(t)=v_n+\frac{t-t_n}{h}(v_n-v_{n-1}),
$$

$$
\int_{t_n}^{t_n+h}p(t)dt=h\left(\frac32v_n-\frac12v_{n-1}\right),
$$

$$
x_{n+1}=x_n+h\left(\frac32v_n-\frac12v_{n-1}\right).
$$

仓库实现还支持非均匀网格；下一节给出其系数。它仍是二阶多步法，但首步 Euler 的误差会保留在整条短轨迹中。



## 非均匀时间网格的正确系数

令当前步长 $h_n=t_{n+1}-t_n$、上一步长 $h_{n-1}=t_n-t_{n-1}$，并令

$$
r=\frac{h_n}{h_{n-1}}.
$$

穿过两个历史点 $(t_{n-1},v_{n-1})$、$(t_n,v_n)$ 的线性插值为

$$
p(t)=v_n+\frac{t-t_n}{h_{n-1}}(v_n-v_{n-1}).
$$

在 $[t_n,t_{n+1}]$ 积分：

$$
x_{n+1}=x_n+h_n\left[\left(1+\frac r2\right)v_n-\frac r2v_{n-1}\right].
$$

当 $r=1$ 才退化为经典的 $3/2,-1/2$。这正是 `solvers/ab2.py` 的实现；因此在 `log_snr` 或其他非均匀 t 网格上，不能错误复用均匀步长系数。


In [ ]:
import torch


class TwoStepAdamsBashforthMethod:
    """Euler warm-up followed by variable-step AB2 with one new NFE per step."""

    def __init__(self, velocity):
        self.velocity = velocity
        self.nfe = 0

    @torch.no_grad()
    def sample(self, x_init, times):
        x = x_init
        previous_velocity = None
        self.nfe = 0
        for index, (t_now, t_next) in enumerate(zip(times[:-1], times[1:])):
            step = t_next - t_now
            velocity_now = self.velocity(x, t_now)
            if index == 0:
                # No v_{-1} exists, so use the required first-order warm-up.
                x = x + step * velocity_now
            else:
                previous_step = t_now - times[index - 1]
                ratio = step / previous_step
                # Exact integral of the linear interpolant through the two velocities.
                x = x + step * ((1.0 + 0.5 * ratio) * velocity_now
                                - 0.5 * ratio * previous_velocity)
            previous_velocity = velocity_now
            self.nfe += 1
        return x


# Constant velocity is exact even on a deliberately nonuniform grid.
times = torch.tensor([0.0, 0.1, 0.35, 0.8, 1.0])
solver = TwoStepAdamsBashforthMethod(lambda x, t: torch.full_like(x, 2.0))
output = solver.sample(torch.zeros(1, 1), times)
torch.testing.assert_close(output, torch.full_like(output, 2.0))
assert solver.nfe == 4
print("nonuniform-grid constant-velocity invariant passed; NFE =", solver.nfe)
